# Create Time Series of Multiple Variables for a Given NERC Region


In [1]:
# Start by importing the packages we need:
import os
import datetime

import pandas as pd
import matplotlib.pyplot as plt


## Set the Directory Structure

In [99]:
# Identify the top-level directory and the subdirectory where the data will be stored:
temp_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/temperature_data/'
load_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/load_data/'
gridview_data_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/gridview_data/'
data_output_dir = '/Users/burl878/Documents/Code/code_repos/burleyson-etal_2026_tbd/data/integrated_time_series/'


## Write a Function to Process the Temperature Time Series Data


In [18]:
# Define a function to process the time series of temperature for a given NERC region:
def process_temperature_time_series(temp_data_dir: str, temp_region: str):
    
    # Read in the raw time series data for all NERC regions:
    temp_df = pd.read_csv((temp_data_dir + 'NERC_Region_Daily_Temperature_1980_to_2024.csv'))
    
    # Subset to just the data for NERC region you want to use:
    subset_df = temp_df[(temp_df['Region'] == temp_region)].copy()

    # Set 'Date' to a datetime variable and sort by date:
    subset_df['Time_UTC'] = pd.to_datetime(subset_df['Date'])
    subset_df = subset_df.sort_values(['Time_UTC'])

    # Add the day of year to be used as an averaging parameter:
    subset_df['DoY'] = subset_df['Time_UTC'].dt.dayofyear

    # Calculate the mean T_Min and T_Max by day of year:
    subset_df['T_Min_Mean'] = subset_df.groupby('DoY')['T_Min'].transform('mean').round(2)
    subset_df['T_Max_Mean'] = subset_df.groupby('DoY')['T_Max'].transform('mean').round(2)
    
    # Only keep the columns we need:
    output_df = subset_df[['Time_UTC','T_Min','T_Min_Mean','T_Max','T_Max_Mean']].copy()
    
    return output_df


In [19]:
# Test the function:
temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, 
                                          temp_region = 'CA')

temp_df


,Time_UTC,T_Min,T_Min_Mean,T_Max,T_Max_Mean
0,1980-01-01,47.43,40.73,56.80,54.30
1,1980-01-02,41.88,41.59,57.35,54.62
2,1980-01-03,43.34,41.77,56.84,54.73
3,1980-01-04,42.60,41.91,55.79,54.62
4,1980-01-05,43.85,42.22,55.37,54.97
...,...,...,...,...,...
16432,2024-12-27,46.73,40.53,55.58,53.38
16433,2024-12-28,47.35,40.89,59.67,53.21
16434,2024-12-29,47.04,41.38,58.75,53.78
16435,2024-12-30,41.68,41.10,57.36,54.05


## Write a Function to Process the Load Time Series Data


In [32]:
def process_load_time_series(load_data_dir: str, load_region: str):

    # Read in the load data and subset to a given year:
    load_df = pd.read_csv((load_data_dir + 'WECC_Hourly_Loads_1980_to_2025.csv'))
       
    # Set 'Time_UTC' to a datetime variable:
    load_df['Time_UTC'] = pd.to_datetime(load_df['Time_UTC'])
    
    # Only keep the columns we need:
    load_df = load_df[['Time_UTC', 'WECC_Load_MWh', (load_region + '_Load_MWh')]].copy()

    # Add the hour of year to be used as an averaging parameter:
    load_df['HoY'] = (((load_df['Time_UTC'].dt.dayofyear -1) * 24) + load_df['Time_UTC'].dt.hour)
    
    # Rename the columns because I'm OCD:
    load_df.rename(columns={(load_region + '_Load_MWh'): 'Region_Load_MWh'}, inplace=True)

    # Calculate the mean load by hour of year:
    load_df['WECC_Load_Mean_MWh'] = load_df.groupby('HoY')['WECC_Load_MWh'].transform('mean').round(2)
    load_df['Region_Load_Mean_MWh'] = load_df.groupby('HoY')['Region_Load_MWh'].transform('mean').round(2)

    # Only keep the columns we need:
    output_df = load_df[['Time_UTC','WECC_Load_MWh','WECC_Load_Mean_MWh','Region_Load_MWh','Region_Load_Mean_MWh']].copy()
    
    return output_df
    

In [66]:
# Test the function:
load_df = process_load_time_series(load_data_dir = load_data_dir, 
                                   load_region = 'CA')

load_df


,Time_UTC,WECC_Load_MWh,WECC_Load_Mean_MWh,Region_Load_MWh,Region_Load_Mean_MWh
0,1980-01-01 00:00:00,95949.05,96181.19,34056.59,32076.15
1,1980-01-01 01:00:00,100408.36,101770.50,35669.03,34548.81
2,1980-01-01 02:00:00,102709.84,104813.27,36599.53,35912.07
3,1980-01-01 03:00:00,104593.05,106864.34,37876.25,37291.22
4,1980-01-01 04:00:00,104588.55,106979.44,38507.77,38252.27
...,...,...,...,...,...
395275,2024-12-31 19:00:00,97933.90,98306.31,33235.87,32963.92
395276,2024-12-31 20:00:00,97279.37,97801.60,33197.71,32993.46
395277,2024-12-31 21:00:00,97107.29,97550.03,33381.65,33155.17
395278,2024-12-31 22:00:00,NaN,97643.19,NaN,33408.53


## Write a Function to Process the Load Shed Time Series Data


In [49]:
def process_load_shed_time_series(gridview_data_dir: str, load_shed_region: str):

    # Read in the load data and subset to a given year:
    load_shed_df = pd.read_csv((gridview_data_dir + 'all_' + load_shed_region + '_load_shed.csv'))

    # Rename the columns:
    load_shed_df.rename(columns={load_shed_df.columns[0]: 'Time_UTC'}, inplace=True)
    load_shed_df.rename(columns={load_shed_df.columns[1]: 'Load_Shed_MWh'}, inplace=True)

    # Set 'Time_UTC' to a datetime variable:
    load_shed_df['Time_UTC'] = pd.to_datetime(load_shed_df['Time_UTC'])

    # Round off the load shed values:
    load_shed_df['Load_Shed_MWh'] = load_shed_df['Load_Shed_MWh'].round(2)
    
    return load_shed_df
    

In [51]:
# Test the function:
load_shed_df = process_load_shed_time_series(gridview_data_dir = gridview_data_dir, 
                                             load_shed_region = 'CA')

load_shed_df


,Time_UTC,Load_Shed_MWh
0,1982-01-01 00:00:00,0.0
1,1982-01-01 01:00:00,0.0
2,1982-01-01 02:00:00,0.0
3,1982-01-01 03:00:00,0.0
4,1982-01-01 04:00:00,0.0
...,...,...
332875,2019-12-31 19:00:00,0.0
332876,2019-12-31 20:00:00,0.0
332877,2019-12-31 21:00:00,0.0
332878,2019-12-31 22:00:00,0.0


## Write a Function to Process the Generation Time Series Data


In [62]:
def process_generation_time_series(gridview_data_dir: str, generation_region: str):

    # Read in the load data and subset to a given year:
    gen_df = pd.read_csv((gridview_data_dir + 'all_' + generation_region + '_generation.csv'))

    # Rename the date column:
    gen_df.rename(columns={'Unnamed: 0': 'Time_UTC'}, inplace=True)
    
    # Extract the month and date for grouping:
    gen_df['Time_UTC'] = pd.to_datetime(gen_df['Time_UTC'])
    
    # Round off the generation values to a single decimal:
    gen_df['Coal'] = gen_df['Coal'].round(1)
    gen_df['Gas'] = gen_df['Gas'].round(1)
    gen_df['Hydro'] = gen_df['Hydro'].round(1)
    gen_df['Other'] = gen_df['Other'].round(1)
    gen_df['Solar'] = gen_df['Solar'].round(1)
    gen_df['Wind'] = gen_df['Wind'].round(1)
    gen_df['Imports'] = gen_df['Imports'].round(1)

    return gen_df
    

In [61]:
# Test the function:
generation_df = process_generation_time_series(gridview_data_dir = gridview_data_dir, 
                                               generation_region = 'CA')

generation_df


,Time_UTC,Coal,Gas,Hydro,Other,Solar,Wind,Imports
0,1982-01-01 00:00:00,55.0,7990.3,4516.7,6705.7,0.0,1384.4,13168.7
1,1982-01-01 01:00:00,55.0,8132.8,4518.6,6515.1,0.0,1524.9,12055.7
2,1982-01-01 02:00:00,55.0,8906.9,4449.7,5312.8,0.0,1232.8,12271.9
3,1982-01-01 03:00:00,55.0,8835.2,4437.3,5514.1,0.0,1130.5,12867.6
4,1982-01-01 04:00:00,55.0,8651.6,4416.4,5378.4,0.0,1318.2,12982.6
...,...,...,...,...,...,...,...,...
332875,2019-12-31 19:00:00,55.0,13289.9,6648.7,7450.7,0.0,2045.4,14621.8
332876,2019-12-31 20:00:00,55.0,12322.2,6642.0,7453.4,0.0,1982.6,13942.3
332877,2019-12-31 21:00:00,55.0,11660.7,6627.2,6010.9,0.0,1973.8,12968.0
332878,2019-12-31 22:00:00,55.0,11668.3,6616.4,5752.6,0.0,2017.1,13034.1


## Create the Integrated Time Series by Merging the Data Streams Together


In [109]:
# Define a function to process the integrated time series for a given NERC region:
def process_integrated_time_series(region: str, load_data_dir: str, temp_data_dir: str, gridview_data_dir: str, data_output_dir: str):

    # Homogenize the region names:
    if region == 'GB':
       load_shed_region = 'BS'
       generation_region = 'BS'
    elif region == 'PNW':
       load_shed_region = 'NW'
       generation_region = 'NW'
    else:
       load_shed_region = region
       generation_region = region 
    
    # Process the load data:
    load_df = process_load_time_series(load_data_dir = load_data_dir, load_region = region)
    load_df['Date'] = load_df['Time_UTC'].dt.date
    load_df['Date'] = pd.to_datetime(load_df['Date'])
    
    # Process the temperature data and rename the date variable:
    temp_df = process_temperature_time_series(temp_data_dir = temp_data_dir, temp_region = region)
    temp_df.rename(columns={'Time_UTC': 'Date'}, inplace=True)

    # Merge the two dataframes together based on common times:
    output_df = load_df.merge(temp_df, on=['Date'], how='left')

    # Process the load shed data:
    load_shed_df = process_load_shed_time_series(gridview_data_dir = gridview_data_dir, load_shed_region = load_shed_region)

    # Merge the load shed data into the output dataframe based on common times:
    output_df = output_df.merge(load_shed_df, on=['Time_UTC'], how='left')

    # Process the generation data:
    generation_df = process_generation_time_series(gridview_data_dir = gridview_data_dir, generation_region = generation_region)

    # Merge the generation data into the output dataframe based on common times:
    output_df = output_df.merge(generation_df, on=['Time_UTC'], how='left')

    # Strip the units from the column names for simplicity:
    output_df.rename(columns={'WECC_Load_MWh': 'WECC_Load', 
                              'WECC_Load_Mean_MWh': 'WECC_Load_Mean',
                              'Region_Load_MWh': 'Region_Load',
                              'Region_Load_Mean_MWh': 'Region_Load_Mean',
                              'Load_Shed_MWh': 'Load_Shed'}, inplace=True) 
    
    # Rearrange the columns:
    output_df = output_df[['Time_UTC','T_Min','T_Min_Mean','T_Max','T_Max_Mean','WECC_Load','WECC_Load_Mean','Region_Load','Region_Load_Mean',
                           'Coal','Gas','Hydro','Other','Solar','Wind','Imports','Load_Shed']].copy()

    # Set the output filename:
    output_filename = (region + '_Integrated_Time_Series_1980_to_2024.csv')
        
    # Write out the dataframe to a .csv file:
    output_df.to_csv((os.path.join(data_output_dir, output_filename)), sep=',', index=False)
    
    return output_df


In [114]:
# Test the function:
integrated_df = process_integrated_time_series(region = 'PNW',
                                               load_data_dir = load_data_dir,
                                               temp_data_dir = temp_data_dir,
                                               gridview_data_dir = gridview_data_dir,
                                               data_output_dir = data_output_dir)

integrated_df


,Time_UTC,T_Min,T_Min_Mean,T_Max,T_Max_Mean,WECC_Load,WECC_Load_Mean,Region_Load,Region_Load_Mean,Coal,Gas,Hydro,Other,Solar,Wind,Imports,Load_Shed
0,1980-01-01 00:00:00,34.58,24.82,39.13,33.76,95949.05,96181.19,22629.90,24049.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1980-01-01 01:00:00,34.58,24.82,39.13,33.76,100408.36,101770.50,23480.00,25069.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1980-01-01 02:00:00,34.58,24.82,39.13,33.76,102709.84,104813.27,23855.15,25758.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1980-01-01 03:00:00,34.58,24.82,39.13,33.76,104593.05,106864.34,24162.33,26175.03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1980-01-01 04:00:00,34.58,24.82,39.13,33.76,104588.55,106979.44,24018.17,25832.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395275,2024-12-31 19:00:00,25.22,25.69,33.85,34.38,97933.90,98306.31,25811.46,25890.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395276,2024-12-31 20:00:00,25.22,25.69,33.85,34.38,97279.37,97801.60,25500.93,25641.79,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395277,2024-12-31 21:00:00,25.22,25.69,33.85,34.38,97107.29,97550.03,25230.91,25415.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
395278,2024-12-31 22:00:00,25.22,25.69,33.85,34.38,NaN,97643.19,NaN,25260.13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [111]:
# Loop over the NERC TPL-008-1 regions in the WECC and process the time series for each one:
for region in ['CA', 'GB', 'PNW', 'RM', 'SW']:
    process_integrated_time_series(region = region,
                                   load_data_dir = load_data_dir,
                                   temp_data_dir = temp_data_dir,
                                   gridview_data_dir = gridview_data_dir,
                                   data_output_dir = data_output_dir)
